[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week02.ipynb)

# Week 2: Tensors & Data Representation
**Introduction to Deep Learning (HIT)** &middot; Part I: Foundations

Tensor operations, shapes, broadcasting, devices; representing images, text, and tabular data as tensors.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Manipulate tensors fluently (reshape, permute, reduce, index).
- Reason about shapes, broadcasting, and device placement.
- Encode real data into correctly shaped tensors.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. Shape, dtype, device
Three properties to check whenever something breaks.

In [ ]:
t = torch.arange(24)
print("shape", tuple(t.shape), "| dtype", t.dtype, "| device", t.device)
t = t.reshape(2, 3, 4)
print("reshaped", tuple(t.shape))
print("permute(2,0,1)", tuple(t.permute(2, 0, 1).shape))
print("t[0,1] ->", t[0, 1].tolist())
print("t[:,:,0] ->", tuple(t[:, :, 0].shape))
print("sum over last dim ->", tuple(t.sum(dim=-1).shape))

## 2. Reductions and the dim argument
sum / mean / max collapse a chosen axis; keepdim keeps it as size 1.

In [ ]:
m = torch.arange(12.0).reshape(3, 4)
print("full mean:", m.mean().item())
print("mean over rows (dim=0):", m.mean(dim=0).tolist())
print("mean over cols (dim=1):", m.mean(dim=1).tolist())
print("max over cols:", m.max(dim=1).values.tolist(), "at idx", m.max(dim=1).indices.tolist())
print("keepdim shape:", tuple(m.sum(dim=1, keepdim=True).shape), "(useful for broadcasting back)")

## 3. Views vs copies
A view shares memory; changing it changes the original.

In [ ]:
a = torch.arange(6)
b = a.view(2, 3)        # shares memory
b[0, 0] = 99
print("a after editing its view:", a.tolist())   # a is changed too
c = a.reshape(3, 2).clone()  # clone breaks the link
c[0, 0] = -1
print("a after editing a clone:", a.tolist())

## 4. Matrix multiply and batched matmul
@ is matrix multiply; for stacks of matrices use it with a batch dimension.

In [ ]:
A = torch.randn(2, 3); B = torch.randn(3, 4)
print("(2,3) @ (3,4) ->", tuple((A @ B).shape))
# a batch of 5 matrix multiplies at once:
batchA = torch.randn(5, 2, 3); batchB = torch.randn(5, 3, 4)
print("batched (5,2,3) @ (5,3,4) ->", tuple((batchA @ batchB).shape))
# einsum spells the indices out explicitly:
print("einsum matches @:", torch.allclose(torch.einsum('ij,jk->ik', A, B), A @ B))

## 5. Broadcasting
Shapes are compared from the trailing dimension; a size of 1 expands.

In [ ]:
a = torch.ones(3, 1); b = torch.ones(1, 4)
print("(3,1) + (1,4) ->", tuple((a + b).shape))          # 3x4
row = torch.arange(4.0)
mat = torch.zeros(3, 4)
print("matrix + row ->", (mat + row).tolist())           # row added to each row

**Predict before running:** what is the result shape of `torch.ones(5,1,4) + torch.ones(3,1)`? Then run it.

In [ ]:
print(tuple((torch.ones(5, 1, 4) + torch.ones(3, 1)).shape))   # broadcasting puzzle

## 6. Normalize with broadcasting
Subtract a per-column mean and divide by a per-column std, no loops.

In [ ]:
data = torch.randn(100, 3) * torch.tensor([5.0, 1.0, 20.0]) + torch.tensor([10.0, -3.0, 0.0])
normed = (data - data.mean(0)) / data.std(0)
print("per-column mean before:", data.mean(0).round(decimals=2).tolist())
print("per-column mean after :", normed.mean(0).round(decimals=2).tolist(), "(~0)")
print("per-column std after  :", normed.std(0).round(decimals=2).tolist(), "(~1)")

## 7. Boolean masks and fancy indexing
Select elements by condition or by an index tensor.

In [ ]:
v = torch.arange(10)
print("v > 5 ->", (v > 5).tolist())
print("v[v > 5] ->", v[v > 5].tolist())
idx = torch.tensor([0, 0, 9, 5])
print("v[[0,0,9,5]] ->", v[idx].tolist(), "(gather by index)")

## 8. A shape-mismatch error, then the fix
Read the error: it names the incompatible dimensions.

In [ ]:
x = torch.ones(3, 4); w = torch.ones(5)   # wrong trailing size
try:
    x + w
except RuntimeError as e:
    print("ERROR:", str(e).splitlines()[0])
w = torch.ones(4)                          # fix: matches trailing dim 4
print("fixed (3,4)+(4,) ->", tuple((x + w).shape))

## 9. Encoding real data as tensors
An image becomes (C, H, W); a table becomes (rows, features).

In [ ]:
import numpy as np
img = np.random.randint(0, 256, size=(8, 8, 3), dtype=np.uint8)   # H, W, C
img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0     # -> C, H, W in [0,1]
print("image tensor:", tuple(img_t.shape), img_t.dtype, "range", img_t.min().item(), img_t.max().item())

table = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
print("table tensor:", tuple(table.shape))
img_t, table = img_t.to(device), table.to(device)
print("moved to", img_t.device)

### Mini-exercise
Given class labels `y = torch.tensor([2, 0, 1, 2])` and 3 classes, build the one-hot matrix of shape (4, 3) without a loop. Hint: `scatter_` or index a 3x3 identity.

*Try it before revealing the solution below.*

**Solution.**

In [ ]:
y = torch.tensor([2, 0, 1, 2])
onehot = torch.eye(3)[y]                       # index the identity matrix
print(onehot.tolist())
# equivalent with scatter:
oh2 = torch.zeros(4, 3); oh2.scatter_(1, y.unsqueeze(1), 1.0)
print("scatter matches:", torch.equal(onehot, oh2))

**Recap:** shape, dtype, device are the first things to check. Broadcasting aligns from the trailing axis. Know when an op views versus copies memory. Matrix multiply and reductions are the workhorses.

---
Student materials for this week: the lab handout (`labs/week02.html`) and the curated references (`references/week02.html`) in the course site.